<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/Stable-Audio-3_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
<a href="https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/Stable-Audio-3_Colab.ipynb" target="_parent"><img src="https://img.shields.io/badge/GitHub-View_Source-181717?logo=github" alt="View on GitHub"/></a>

# 🔊 Stable Audio 3 — Ultra-Fast Stereo SFX, Audio Editing & Windows Sound Pack Studio

<p align="left">
  <a href="https://github.com/Stability-AI/stable-audio-3"><img src="https://img.shields.io/badge/GitHub-Stability--AI%2Fstable--audio--3-181717?logo=github" alt="GitHub"></a>
  <a href="https://huggingface.co/collections/stabilityai/stable-audio-3"><img src="https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Stable%20Audio%203-amber" alt="HuggingFace"></a>
  <img src="https://img.shields.io/badge/Audio-44.1%20kHz%20Stereo%20(SAME%20VAE)-blue" alt="44.1kHz Stereo">
  <img src="https://img.shields.io/badge/Colab-T4%20%2F%20L4%20%2F%20A100%20Ready-orange?logo=googlecolab" alt="Colab Ready">
</p>

Welcome to the all-in-one **Stable Audio 3** Google Colab studio! Built on Stability AI's [`Stability-AI/stable-audio-3`](https://github.com/Stability-AI/stable-audio-3) platform and the **SAME (Semantic-Acoustic Music Encoder)** 44.1 kHz stereo latent space, this notebook provides:

- **⚡ Dedicated `small-sfx` (433M) & `medium` (1.4B) Models** — Adversarially post-trained to synthesize variable-length **44.1 kHz stereo** sound effects and music in just **8 steps** (~0.5 seconds on GPU, < 2 GB VRAM for `small-sfx`).
- **🎛️ Base Checkpoints (`small-sfx-base`, `medium-base`) + Diffusers Fallback** — Switch to 50-step `-base` models for full Classifier-Free Guidance (`cfg_scale`) and `negative_prompt` control, or use the `diffusers` `StableAudioPipeline` (`stable-audio-open-small`, `stable-audio-open-1.0`) checkpoints.
- **🔄 Audio-to-Audio Style Variation** — Upload any existing `.wav` recording or classic OS chime and morph its timbre/genre using `init_audio` and `init_noise_level`.
- **🩹 Multi-Region Audio Inpainting & Continuation** — Regenerate one or multiple non-contiguous time windows (`inpaint_mask_start_seconds` / `inpaint_mask_end_seconds`) or extend a sound effect seamlessly.
- **🪟 Windows 10/11 Sound Event Pack Studio (`AppEvents\Schemes`)** — Batch-generate up to **21 canonical Windows system events** across 8 curated sonic themes with **zero-latency spectral-flux onset auto-trim**, phase-safe stereo mastering, an **inline HTML5 WebAudio Audition Soundboard**, and a 1-click **`Install_*.ps1` + `Uninstall_*.ps1` + `.reg` ZIP archive**.
- **🧬 Runtime LoRA Adapter Manager** — Load and blend custom Stable Audio 3 `.ckpt` / `.safetensors` LoRA weights (`load_lora`, `set_lora_strength`).

In [ ]:
#@title Step 1: Verify GPU & Install Stable Audio 3 Dependencies
import os
import sys
import subprocess
import torch
from tqdm.auto import tqdm
from IPython.display import clear_output

REPO_DIR = "/content/stable-audio-3"

def run_cmd(cmd, check=False):
    print(f"⏳ Running: {cmd}")
    return subprocess.run(cmd, shell=True, check=check)

try:
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        print(f"✅ GPU Detected: {gpu_name} ({vram_gb:.1f} GB VRAM)")
    else:
        print("⚠️ No CUDA GPU detected — Stable Audio 3 'small-sfx' still runs on CPU!")

    if not os.path.exists(REPO_DIR):
        run_cmd(f"git clone --depth 1 https://github.com/Stability-AI/stable-audio-3.git {REPO_DIR}", check=True)

    run_cmd("apt-get update -qq && apt-get install -y -qq ffmpeg libsndfile1", check=False)

    pip_pkgs = (
        "einops>=0.8.2 einops-exts>=0.0.4 safetensors>=0.4.5 soundfile>=0.12.1 "
        "huggingface_hub>=0.26.0 transformers>=4.48.0 accelerate>=1.2.0 "
        "gradio>=5.0.0 matplotlib scipy librosa torchaudio diffusers tqdm"
    )
    run_cmd(f"{sys.executable} -m pip install -q -U {pip_pkgs}", check=False)
    run_cmd(f"{sys.executable} -m pip install -q --no-deps -e {REPO_DIR}", check=False)

    try:
        import flash_attn
        print("✅ FlashAttention-2 is already installed.")
    except Exception:
        run_cmd(f"{sys.executable} -m pip install -q flash-attn --no-build-isolation", check=False)

    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

    clear_output()
    print("✅ Step 1 Complete: Stable Audio 3 repository and audio dependencies are installed!")
except Exception as e:
    print(f"❌ Installation Error: {e}")
    raise

In [ ]:
#@title Step 2: Mount Google Drive & Configure Persistent Model Cache
import os
import sys

WORKSPACE_DIR = "/content/stable_audio_3_ws"
os.makedirs(WORKSPACE_DIR, exist_ok=True)

DRIVE_MOUNTED = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_MOUNTED = os.path.ismount("/content/drive")
except Exception as e:
    print(f"ℹ️ Google Drive not mounted (local or non-Colab session): {e}")

if DRIVE_MOUNTED:
    CACHE_ROOT = "/content/drive/MyDrive/AEI_Audio_Cache/stable_audio_3"
    OUTPUT_DIR = "/content/drive/MyDrive/AEI_Audio_Exports/Stable_Audio_3"
    SFX_LIB_DIR = "/content/drive/MyDrive/AEI_SFX_Library"
else:
    CACHE_ROOT = os.path.join(WORKSPACE_DIR, "cache")
    OUTPUT_DIR = os.path.join(WORKSPACE_DIR, "outputs")
    SFX_LIB_DIR = os.path.join(WORKSPACE_DIR, "sfx_library")

for d in (CACHE_ROOT, OUTPUT_DIR, SFX_LIB_DIR):
    os.makedirs(d, exist_ok=True)

os.environ["HF_HOME"] = os.path.join(CACHE_ROOT, "huggingface")
os.environ["TORCH_HOME"] = os.path.join(CACHE_ROOT, "torch")
os.environ.setdefault("TORCHDYNAMO_DISABLE", "1")
os.makedirs(os.environ["HF_HOME"], exist_ok=True)

try:
    from google.colab import userdata
    secret_tok = userdata.get("HF_TOKEN")
    if secret_tok:
        os.environ["HF_TOKEN"] = secret_tok.strip()
        print("🔑 Loaded HF_TOKEN from Colab Secrets.")
except Exception:
    pass

print(f"📂 HF_HOME Cache : {os.environ['HF_HOME']}")
print(f"📂 Output Exports: {OUTPUT_DIR}")

In [ ]:
#@title Step 3: Core Stable Audio 3 Engine & Windows Sound Packager
import os
import gc
import sys
import time
import glob
import json
import torch
import torchaudio
import numpy as np
import soundfile as sf
from huggingface_hub import hf_hub_download, login

REPO_DIR = "/content/stable-audio-3"
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# ==============================================================================
# Shared Windows 10/11 Sound Scheme & Zero-Latency UI Mastering Engine
# ==============================================================================
import io
import re
import json
import uuid
import base64
import shutil
import zipfile
from datetime import datetime
import scipy.signal
import numpy as np
import soundfile as sf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

WINDOWS_SYSTEM_EVENTS = {
    "WindowsLogon": {
        "label": "Windows Logon / Startup",
        "reg_app": ".Default",
        "duration": 3.0,
        "contour": "warm welcoming ascending chord swell with a gentle shimmering reverb tail",
    },
    "WindowsLogoff": {
        "label": "Windows Logoff / Shutdown",
        "reg_app": ".Default",
        "duration": 2.5,
        "contour": "peaceful resolving descending harmonic chord that fades smoothly into silence",
    },
    "WindowsUnlock": {
        "label": "Windows Unlock / Sign-In",
        "reg_app": ".Default",
        "duration": 1.4,
        "contour": "bright welcoming two-note ascending unlock chime with airy resonance",
    },
    "SystemNotification": {
        "label": "System Notification (Toast)",
        "reg_app": ".Default",
        "duration": 1.2,
        "contour": "crisp pleasant two-note notification chime, clear transient attack, short clean tail",
    },
    "WindowsNotifySystemGeneric": {
        "label": "Action Center / Generic Toast Banner",
        "reg_app": ".Default",
        "duration": 1.2,
        "contour": "modern crystalline two-tone notification ping with soft harmonic decay",
    },
    "SystemAsterisk": {
        "label": "Asterisk / Info Pop-up",
        "reg_app": ".Default",
        "duration": 1.0,
        "contour": "soft single crystalline ping with subtle harmonic resonance",
    },
    "SystemExclamation": {
        "label": "Exclamation / Warning Alert",
        "reg_app": ".Default",
        "duration": 1.2,
        "contour": "attention-grabbing mid-range double pulse chime, alert but non-harsh",
    },
    "SystemHand": {
        "label": "Critical Stop / Error",
        "reg_app": ".Default",
        "duration": 1.2,
        "contour": "low muted percussive double thud with a tense minor interval, error alert",
    },
    "SystemQuestion": {
        "label": "Question / Confirmation Dialog",
        "reg_app": ".Default",
        "duration": 1.0,
        "contour": "gentle inquisitive rising two-note harmonic inflection chime",
    },
    "Default": {
        "label": "Default Beep / Fallback Alert",
        "reg_app": ".Default",
        "duration": 0.8,
        "contour": "clean short balanced harmonic ping, neutral interface alert",
    },
    "DeviceConnect": {
        "label": "Device Connect (USB Plug-In)",
        "reg_app": ".Default",
        "duration": 1.0,
        "contour": "upbeat two-tone ascending interval (low note to higher fifth), hardware connected",
    },
    "DeviceDisconnect": {
        "label": "Device Disconnect (USB Unplug)",
        "reg_app": ".Default",
        "duration": 1.0,
        "contour": "smooth two-tone descending interval (high note to lower root), hardware unplugged",
    },
    "DeviceFail": {
        "label": "Device Failed to Connect",
        "reg_app": ".Default",
        "duration": 1.2,
        "contour": "short stuttering muted low buzz double tap, hardware connection failure",
    },
    "EmptyRecycleBin": {
        "label": "Empty Recycle Bin",
        "reg_app": "Explorer",
        "duration": 1.2,
        "contour": "crisp satisfying paper and glass shimmer dissolve whoosh",
    },
    "WindowsUAC": {
        "label": "User Account Control (UAC Prompt)",
        "reg_app": ".Default",
        "duration": 1.2,
        "contour": "authoritative clean two-note security confirmation chime",
    },
    "LowBatteryAlarm": {
        "label": "Low Battery Alarm",
        "reg_app": ".Default",
        "duration": 1.2,
        "contour": "gentle pulsing two-note reminder tone",
    },
    "CriticalBatteryAlarm": {
        "label": "Critical Battery Alarm",
        "reg_app": ".Default",
        "duration": 1.5,
        "contour": "urgent rapid triple pulse warning chime",
    },
    "MailBeep": {
        "label": "New Mail / Message Beep",
        "reg_app": ".Default",
        "duration": 1.0,
        "contour": "bright airy bell ding notification",
    },
    "MessageNudge": {
        "label": "Message Nudge / Chat Ping",
        "reg_app": ".Default",
        "duration": 0.9,
        "contour": "playful double tactile pop ping notification",
    },
    "PrintComplete": {
        "label": "Print / Task Complete",
        "reg_app": ".Default",
        "duration": 1.3,
        "contour": "satisfying upbeat three-note completion flourish chime",
    },
    "MenuCommand": {
        "label": "Menu Command / Tactile UI Click",
        "reg_app": ".Default",
        "duration": 0.6,
        "contour": "ultra-short crisp tactile UI click pop transient, dry and immediate",
    },
}

SONIC_THEME_PRESETS = {
    "Frutiger Aero Crystal Glass": (
        "crystalline water-drop glass marimba and airy harmonic bell pads, "
        "clean high-tech Frutiger Aero operating system UI sound effect, studio quality"
    ),
    "Cyberpunk Neon HUD": (
        "futuristic cyberpunk holographic HUD telemetry synthesizer, "
        "crisp digital micro-transient with warm sub-bass resonance, sci-fi UI sound"
    ),
    "Cozy Wooden Kalimba & Mallet": (
        "warm organic rosewood marimba, thumb piano kalimba, and soft felt mallet percussion, "
        "intimate close-mic acoustic UI sound effect"
    ),
    "Sci-Fi Spaceship Bridge": (
        "clean starship computer console telemetry bleep, precision FM synthesis harmonic tone, "
        "futuristic spacecraft interface sound effect"
    ),
    "Retro 16-Bit Chiptune Arcade": (
        "crisp nostalgic 16-bit FM synth arcade game UI sound effect, "
        "bright square and triangle wave arpeggio, clean retro console audio"
    ),
    "Warm Lo-Fi Analog Tape Synth": (
        "warm vintage analog poly-synth and Rhodes electric piano chime with gentle tape saturation, "
        "cozy lo-fi interface sound effect"
    ),
    "Minimalist Tactile Micro-Foley": (
        "ultra-clean modern minimalist UI micro-foley, soft mechanical switch click and muted glass tap, "
        "tight dry studio recording"
    ),
    "Dark Fantasy Crystal & Harp": (
        "ethereal celesta, delicate harp harmonic pluck, and mystical resonant crystal chime, "
        "fantasy RPG interface sound effect"
    ),
}

DEFAULT_SELECTED_WIN_EVENTS = [
    "WindowsLogon",
    "WindowsLogoff",
    "WindowsUnlock",
    "SystemNotification",
    "WindowsNotifySystemGeneric",
    "SystemAsterisk",
    "SystemExclamation",
    "SystemHand",
    "DeviceConnect",
    "DeviceDisconnect",
    "EmptyRecycleBin",
    "WindowsUAC",
]


def build_windows_event_prompt(event_key: str, theme_name: str, custom_style: str = "") -> str:
    ev = WINDOWS_SYSTEM_EVENTS.get(event_key, WINDOWS_SYSTEM_EVENTS["SystemNotification"])
    style_desc = custom_style.strip() if custom_style and custom_style.strip() else SONIC_THEME_PRESETS.get(
        theme_name, SONIC_THEME_PRESETS["Frutiger Aero Crystal Glass"]
    )
    return f"{ev['contour']}, {style_desc}, high fidelity, no background noise"


def apply_phase_safe_stereo_widener(arr: np.ndarray, sr: int, width: float = 0.22) -> np.ndarray:
    """Upgrade mono (N, 1) to phase-coherent stereo (N, 2) or subtly widen stereo above 350 Hz."""
    if arr.ndim == 1:
        arr = arr[:, None]
    n_samples, ch = arr.shape
    if n_samples < 64 or width <= 0.01:
        return np.repeat(arr, 2, axis=1) if ch == 1 else arr

    nyq = 0.5 * float(sr)
    cutoff = min(max(350.0 / nyq, 0.01), 0.85)
    b, a = scipy.signal.butter(2, cutoff, btype="highpass")

    if ch == 1:
        mono = arr[:, 0]
        high = scipy.signal.lfilter(b, a, mono).astype(np.float32)
        delay_samples = max(1, int(0.00038 * sr))  # ~0.38 ms Haas micro-decorrelation
        side = np.zeros_like(high)
        if delay_samples < n_samples:
            side[delay_samples:] = high[:-delay_samples] - high[delay_samples:]
        side *= float(width)
        left = mono + side
        right = mono - side
        return np.stack([left, right], axis=1).astype(np.float32)
    else:
        mid = 0.5 * (arr[:, 0] + arr[:, 1])
        side = 0.5 * (arr[:, 0] - arr[:, 1])
        side_high = scipy.signal.lfilter(b, a, side).astype(np.float32)
        side_boosted = side + float(width) * 0.5 * side_high
        return np.stack([mid + side_boosted, mid - side_boosted], axis=1).astype(np.float32)


def master_ui_sound(
    audio_np: np.ndarray,
    sr: int,
    target_duration_s: float = None,
    trim_silence: bool = True,
    trim_db: float = -45.0,
    fade_in_ms: float = 1.5,
    fade_out_ms: float = 120.0,
    target_dbfs: float = -1.2,
    stereo_widen: bool = True,
    hpf_hz: float = 40.0,
    pitch_semitones: float = 0.0,
) -> np.ndarray:
    """Zero-latency UI sound mastering: snaps to first transient (<1ms pre-roll), 40Hz sub-rumble filter, optional semitone transpose, 1.5ms click-free micro-fade, phase-safe stereo widening, and true-peak normalization."""
    arr = np.asarray(audio_np, dtype=np.float32)
    if arr.ndim == 1:
        arr = arr[:, None]  # (samples, 1)
    elif arr.ndim == 2 and arr.shape[0] <= 2 and arr.shape[1] > arr.shape[0]:
        arr = arr.T  # (samples, channels)

    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    arr = arr - np.mean(arr, axis=0, keepdims=True)

    # Optional sub-rumble high-pass filter (default 40 Hz) to keep UI chimes crisp
    if hpf_hz and float(hpf_hz) > 10.0 and arr.shape[0] > 64:
        nyq = 0.5 * float(sr)
        cutoff = min(max(float(hpf_hz) / nyq, 0.002), 0.45)
        b_hp, a_hp = scipy.signal.butter(2, cutoff, btype="highpass")
        arr = scipy.signal.lfilter(b_hp, a_hp, arr, axis=0).astype(np.float32)

    # Optional semitone pitch shift via high-precision polyphase resampling
    if pitch_semitones and abs(float(pitch_semitones)) >= 0.05 and arr.shape[0] > 64:
        factor = 2.0 ** (float(pitch_semitones) / 12.0)
        up = 1000
        down = max(100, int(round(1000.0 * factor)))
        gcd = np.gcd(up, down)
        arr = scipy.signal.resample_poly(arr, up // gcd, down // gcd, axis=0).astype(np.float32)

    if trim_silence and arr.shape[0] > int(0.04 * sr):
        mono_env = np.max(np.abs(arr), axis=1)
        diff_env = np.abs(np.diff(mono_env, prepend=mono_env[0]))
        peak = float(np.max(mono_env))
        if peak > 1e-6:
            amp_thresh = peak * (10.0 ** (float(trim_db) / 20.0))
            flux_thresh = float(np.max(diff_env)) * 0.08
            active_idx = np.where((mono_env >= amp_thresh) | (diff_env >= flux_thresh))[0]
            if active_idx.size > 0:
                pre_roll = int(0.0008 * sr)   # 0.8 ms zero-latency pre-roll
                post_roll = int(0.160 * sr)   # 160 ms natural reverb tail retention
                start_i = max(0, int(active_idx[0]) - pre_roll)
                end_i = min(arr.shape[0], int(active_idx[-1]) + post_roll)
                if end_i > start_i + int(0.03 * sr):
                    arr = arr[start_i:end_i]

    if target_duration_s is not None and target_duration_s > 0:
        max_samples = int(float(target_duration_s) * sr)
        if arr.shape[0] > max_samples:
            arr = arr[:max_samples]

    n_samples = arr.shape[0]
    if n_samples == 0:
        return np.zeros((int(0.5 * sr), 2), dtype=np.float32)

    if stereo_widen:
        arr = apply_phase_safe_stereo_widener(arr, sr, width=0.22 if arr.shape[1] == 1 else 0.12)

    fi_samples = min(int((float(fade_in_ms) / 1000.0) * sr), max(1, n_samples // 8))
    if fi_samples > 1:
        fi_curve = 0.5 * (1.0 - np.cos(np.linspace(0.0, np.pi, fi_samples, dtype=np.float32)))
        arr[:fi_samples] *= fi_curve[:, None]

    fo_samples = min(int((float(fade_out_ms) / 1000.0) * sr), max(1, n_samples // 2))
    if fo_samples > 1:
        fo_curve = (0.5 * (1.0 + np.cos(np.linspace(0.0, np.pi, fo_samples, dtype=np.float32)))) ** 1.2
        arr[-fo_samples:] *= fo_curve[:, None]

    peak_val = float(np.max(np.abs(arr)))
    if peak_val > 1e-6:
        target_amp = 10.0 ** (float(target_dbfs) / 20.0)
        arr = arr * (target_amp / peak_val)

    arr = np.clip(arr, -0.999, 0.999)
    return arr if arr.shape[1] > 1 else arr[:, 0]


def save_pcm16_wav(path: str, audio_np: np.ndarray, sr: int, subtype: str = "PCM_16") -> str:
    abs_path = os.path.abspath(path)
    os.makedirs(os.path.dirname(abs_path), exist_ok=True)
    sf.write(abs_path, audio_np, int(sr), subtype=subtype)
    # Mirror to AEI_TTS_Out if Google Drive is mounted so companion notebooks (InfiniteTalk, SeedVR2, Audio_PostProcessor) auto-discover it
    if "/content/drive/MyDrive/" in abs_path and "/AEI_TTS_Out/" not in abs_path:
        try:
            mirror_dir = "/content/drive/MyDrive/AEI_TTS_Out/SFX_Exports"
            os.makedirs(mirror_dir, exist_ok=True)
            shutil.copy2(abs_path, os.path.join(mirror_dir, os.path.basename(abs_path)))
        except Exception:
            pass
    return abs_path


def load_audio_file_safe(audio_path: str, target_sr: int = None, mono: bool = False):
    """Robust audio loader using soundfile first, falling back to librosa/torchaudio."""
    try:
        data, sr = sf.read(audio_path, dtype="float32", always_2d=True)
        # data is (T, C) -> transpose to (C, T)
        wav = data.T
    except Exception:
        import librosa
        wav, sr = librosa.load(audio_path, sr=None, mono=False)
        if wav.ndim == 1:
            wav = wav[None, :]
    if mono and wav.shape[0] > 1:
        wav = np.mean(wav, axis=0, keepdims=True)
    if target_sr is not None and int(sr) != int(target_sr):
        gcd = np.gcd(int(sr), int(target_sr))
        wav = scipy.signal.resample_poly(wav, int(target_sr) // gcd, int(sr) // gcd, axis=-1).astype(np.float32)
        sr = int(target_sr)
    return wav, int(sr)


def render_waveform_and_spectrogram(wav_path: str, title: str = "Audio Signal & Spectrogram") -> str:
    """Render a dual-panel waveform + spectrogram plot for visual transient & tail inspection."""
    try:
        data, sr = sf.read(wav_path, always_2d=True)
        mono = np.mean(data, axis=1)
        dur = len(mono) / float(sr)
        t = np.linspace(0.0, dur, len(mono))

        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8.5, 4.2), facecolor="#0f172a")
        for ax in (ax1, ax2):
            ax.set_facecolor("#1e293b")
            ax.tick_params(colors="#cbd5e1", labelsize=8)
            for spine in ax.spines.values():
                spine.set_color("#334155")

        if data.shape[1] > 1:
            ax1.plot(t, data[:, 0], color="#38bdf8", linewidth=0.8, alpha=0.85, label="L")
            ax1.plot(t, data[:, 1], color="#f472b6", linewidth=0.8, alpha=0.75, label="R")
            ax1.legend(loc="upper right", fontsize=7, facecolor="#0f172a", edgecolor="#334155", labelcolor="#f8fafc")
        else:
            ax1.plot(t, mono, color="#38bdf8", linewidth=0.9)
        ax1.set_xlim(0, max(dur, 0.05))
        ax1.set_ylim(-1.05, 1.05)
        ax1.set_ylabel("Amplitude", color="#e2e8f0", fontsize=8)
        ch_label = "Stereo" if data.shape[1] > 1 else "Mono"
        ax1.set_title(f"{title} ({sr} Hz {ch_label}, {dur:.2f}s)", color="#f8fafc", fontsize=10, fontweight="bold")
        ax1.grid(True, color="#334155", alpha=0.4, linestyle="--")

        nfft = min(1024, max(128, 2 ** int(np.log2(max(len(mono) // 8, 128)))))
        ax2.specgram(mono + 1e-10, NFFT=nfft, Fs=sr, noverlap=nfft // 2, cmap="magma")
        ax2.set_xlim(0, max(dur, 0.05))
        ax2.set_xlabel("Time (seconds)", color="#e2e8f0", fontsize=8)
        ax2.set_ylabel("Freq (Hz)", color="#e2e8f0", fontsize=8)

        plt.tight_layout()
        out_png = os.path.join("/tmp", f"spec_{uuid.uuid4().hex[:8]}.png")
        fig.savefig(out_png, dpi=120, facecolor=fig.get_facecolor())
        plt.close(fig)
        return out_png
    except Exception as e:
        print(f"Spectrogram render warning: {e}")
        return None


def build_audition_soundboard_html(
    scheme_name: str,
    event_wav_map: dict,
    theme_name: str,
    model_label: str,
    embed_base64: bool = False,
) -> str:
    """Build a dark-mode HTML5 WebAudio interactive soundboard with 1-click sequential Play-All auditioning."""
    board_uid = f"sb_{uuid.uuid4().hex[:8]}"
    cards_html = []
    for ev_key, wav_info in event_wav_map.items():
        ev_meta = WINDOWS_SYSTEM_EVENTS.get(ev_key, {"label": ev_key, "reg_app": ".Default", "duration": 1.0})
        if embed_base64 and os.path.exists(wav_info["path"]):
            with open(wav_info["path"], "rb") as bf:
                b64 = base64.b64encode(bf.read()).decode("ascii")
            audio_src = f"data:audio/wav;base64,{b64}"
        else:
            audio_src = f"Media/{wav_info['filename']}"

        cards_html.append(
            f'<div class="aei-sb-card" style="background:#1e293b;border:1px solid #334155;border-radius:10px;padding:10px 12px;display:flex;flex-direction:column;gap:6px;transition:all 0.2s ease;">'
            f'<div style="display:flex;justify-content:space-between;align-items:center;">'
            f'<strong style="color:#f8fafc;font-size:13px;">🔔 {ev_meta.get("label", ev_key)}</strong>'
            f'<span style="background:#0f172a;color:#38bdf8;font-size:11px;padding:2px 6px;border-radius:4px;font-family:monospace;">{ev_meta.get("reg_app", ".Default")}\\{ev_key}</span>'
            f'</div>'
            f'<audio controls preload="none" src="{audio_src}" style="width:100%;height:32px;"></audio>'
            f'</div>'
        )
    grid_str = "\n".join(cards_html)
    play_all_js = (
        f"const root=document.getElementById('{board_uid}');"
        f"if(!root)return;"
        f"const audios=Array.from(root.querySelectorAll('audio'));"
        f"const cards=Array.from(root.querySelectorAll('.aei-sb-card'));"
        f"if(root._playing){{root._playing=false;audios.forEach(a=>{{a.pause();a.currentTime=0;}});cards.forEach(c=>c.style.borderColor='#334155');this.textContent='▶ Play All Events Sequentially';return;}}"
        f"root._playing=true;this.textContent='⏹ Stop Sequence';"
        f"let idx=0;const btn=this;"
        f"const next=()=>{{if(!root._playing||idx>=audios.length){{root._playing=false;btn.textContent='▶ Play All Events Sequentially';cards.forEach(c=>c.style.borderColor='#334155');return;}}"
        f"cards.forEach((c,i)=>{{c.style.borderColor=(i===idx)?'#38bdf8':'#334155';}});"
        f"const a=audios[idx++];a.currentTime=0;"
        f"a.onended=()=>{{setTimeout(next,280);}};"
        f"a.onerror=()=>{{setTimeout(next,100);}};"
        f"a.play().catch(()=>setTimeout(next,100));}};"
        f"next();"
    )
    return (
        f'<div id="{board_uid}" style="background:#0f172a;color:#f8fafc;padding:16px;border-radius:12px;border:1px solid #334155;font-family:system-ui,-apple-system,sans-serif;">'
        f'<div style="display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:10px;margin-bottom:12px;">'
        f'<div>'
        f'<h3 style="margin:0 0 4px 0;color:#38bdf8;font-size:16px;">🪟 {scheme_name} — Interactive Windows 10/11 Soundboard</h3>'
        f'<p style="margin:0;color:#94a3b8;font-size:12px;">Theme: <b>{theme_name}</b> · Engine: <b>{model_label}</b> · Events: <b>{len(event_wav_map)}</b></p>'
        f'</div>'
        f'<button type="button" onclick="{play_all_js}" style="background:#0284c7;color:#ffffff;border:none;border-radius:8px;padding:8px 14px;font-size:12px;font-weight:600;cursor:pointer;">▶ Play All Events Sequentially</button>'
        f'</div>'
        f'<div style="display:grid;grid-template-columns:repeat(auto-fill,minmax(270px,1fr));gap:10px;">'
        f'{grid_str}'
        f'</div>'
        f'</div>'
    )


LAST_WINPACK_STATE = {
    "pack_dir": None,
    "scheme_name": "AEI_Custom_Scheme",
    "theme_name": "Frutiger Aero Crystal Glass",
    "model_label": "AEI Sound Studio",
    "output_dir": "/content",
    "event_wav_map": {},
}


def package_windows_sound_scheme_zip(
    scheme_name: str,
    event_wav_map: dict,
    theme_name: str,
    model_label: str,
    output_dir: str,
) -> str:
    """Create a complete Windows 10/11 Sound Scheme ZIP with .wav files, Install/Uninstall .ps1, .reg file, Scheme.ini, soundpack_manifest.json, and audition_board.html."""
    safe_id = re.sub(r"[^A-Za-z0-9_-]+", "_", (scheme_name or "AEI_Custom_Sounds").strip())[:32].strip("_") or "AEI_Custom"
    display_name = (scheme_name or "AEI Custom Sounds").strip()[:48]
    os.makedirs(output_dir, exist_ok=True)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    zip_path = os.path.join(output_dir, f"{safe_id}_Windows_SoundPack_{ts}.zip")

    ps1_lines = [
        f"# ==============================================================================",
        f"# Windows 10/11 Sound Scheme Automated Installer: {display_name}",
        f"# Generated by AEI Colab Notebooks ({model_label})",
        f"# ==============================================================================",
        f"$ErrorActionPreference = 'Stop'",
        f"$SchemeId = '{safe_id}'",
        f"$SchemeDisplay = '{display_name}'",
        f"$TargetDir = Join-Path $env:LOCALAPPDATA \"WindowsSoundSchemes\\$SchemeId\"",
        f"New-Item -ItemType Directory -Force -Path $TargetDir | Out-Null",
        f"$BackupReg = Join-Path $TargetDir 'Backup_Previous_AppEvents.reg'",
        f"if (-not (Test-Path $BackupReg)) {{ reg export 'HKCU\\AppEvents\\Schemes' $BackupReg /y | Out-Null }}",
        f"Copy-Item -Path (Join-Path $PSScriptRoot 'Media\\*.wav') -Destination $TargetDir -Force",
        f"",
        f"$NamesKey = \"HKCU:\\AppEvents\\Schemes\\Names\\$SchemeId\"",
        f"New-Item -Path $NamesKey -Force | Out-Null",
        f"Set-ItemProperty -Path $NamesKey -Name '(Default)' -Value $SchemeDisplay",
        f"",
        f"$EventMap = @{{",
    ]
    for ev_key, wav_info in event_wav_map.items():
        ev_meta = WINDOWS_SYSTEM_EVENTS.get(ev_key, {"reg_app": ".Default"})
        ps1_lines.append(f"    '{ev_meta['reg_app']}\\{ev_key}' = '{wav_info['filename']}'")
    ps1_lines += [
        f"}}",
        f"$FirstWav = $null",
        f"foreach ($entry in $EventMap.GetEnumerator()) {{",
        f"    $wavPath = Join-Path $TargetDir $entry.Value",
        f"    if (-not $FirstWav -and (Test-Path $wavPath)) {{ $FirstWav = $wavPath }}",
        f"    $schemeEventKey = \"HKCU:\\AppEvents\\Schemes\\Apps\\$($entry.Key)\\$SchemeId\"",
        f"    $currentEventKey = \"HKCU:\\AppEvents\\Schemes\\Apps\\$($entry.Key)\\.Current\"",
        f"    New-Item -Path $schemeEventKey -Force | Out-Null",
        f"    Set-ItemProperty -Path $schemeEventKey -Name '(Default)' -Value $wavPath",
        f"    New-Item -Path $currentEventKey -Force | Out-Null",
        f"    Set-ItemProperty -Path $currentEventKey -Name '(Default)' -Value $wavPath",
        f"}}",
        f"Set-ItemProperty -Path 'HKCU:\\AppEvents\\Schemes' -Name '(Default)' -Value $SchemeId",
        f"if ($FirstWav) {{ try {{ (New-Object System.Media.SoundPlayer $FirstWav).Play() }} catch {{}} }}",
        f"Write-Host \"Successfully installed and activated Windows Sound Scheme: $SchemeDisplay\" -ForegroundColor Green",
        f"Write-Host \"Prior registry state backed up to: $BackupReg\" -ForegroundColor Cyan",
        f"Start-Process 'control.exe' -ArgumentList 'mmsys.cpl,,2'",
    ]

    uninstall_ps1 = [
        f"# ==============================================================================",
        f"# Windows 10/11 Sound Scheme Uninstaller & Default Restore: {display_name}",
        f"# ==============================================================================",
        f"$ErrorActionPreference = 'SilentlyContinue'",
        f"$SchemeId = '{safe_id}'",
        f"Set-ItemProperty -Path 'HKCU:\\AppEvents\\Schemes' -Name '(Default)' -Value '.Default'",
        f"Get-ChildItem 'HKCU:\\AppEvents\\Schemes\\Apps' -Recurse | Where-Object {{ $_.PSChildName -eq $SchemeId }} | Remove-Item -Recurse -Force",
        f"Remove-Item -Path \"HKCU:\\AppEvents\\Schemes\\Names\\$SchemeId\" -Recurse -Force",
        f"$TargetDir = Join-Path $env:LOCALAPPDATA \"WindowsSoundSchemes\\$SchemeId\"",
        f"if (Test-Path $TargetDir) {{ Remove-Item -Path $TargetDir -Recurse -Force }}",
        f"Write-Host \"Restored Windows Default sound scheme and removed $SchemeId.\" -ForegroundColor Yellow",
        f"Start-Process 'control.exe' -ArgumentList 'mmsys.cpl,,2'",
    ]

    reg_lines = [
        "Windows Registry Editor Version 5.00",
        "",
        f"; Windows Sound Scheme: {display_name} ({model_label})",
        f"[HKEY_CURRENT_USER\\AppEvents\\Schemes\\Names\\{safe_id}]",
        f"@=\"{display_name}\"",
        "",
    ]
    for ev_key, wav_info in event_wav_map.items():
        ev_meta = WINDOWS_SYSTEM_EVENTS.get(ev_key, {"reg_app": ".Default"})
        escaped_path = f"C:\\\\Windows\\\\Media\\\\{safe_id}\\\\{wav_info['filename']}"
        reg_lines += [
            f"[HKEY_CURRENT_USER\\AppEvents\\Schemes\\Apps\\{ev_meta['reg_app']}\\{ev_key}\\{safe_id}]",
            f"@=\"{escaped_path}\"",
            "",
            f"[HKEY_CURRENT_USER\\AppEvents\\Schemes\\Apps\\{ev_meta['reg_app']}\\{ev_key}\\.Current]",
            f"@=\"{escaped_path}\"",
            "",
        ]

    ini_lines = [
        "[Scheme]",
        f"Name={display_name}",
        f"SchemeId={safe_id}",
        f"Theme={theme_name}",
        f"Generator={model_label}",
        f"CreatedAt={datetime.now().isoformat()}",
        "",
        "[Events]",
    ]
    for ev_key, wav_info in event_wav_map.items():
        ini_lines.append(f"{ev_key}=Media/{wav_info['filename']}")
        ini_lines.append(f"{ev_key}_Prompt={wav_info.get('prompt', '')}")

    manifest_data = {
        "scheme_name": display_name,
        "scheme_id": safe_id,
        "theme_name": theme_name,
        "generator": model_label,
        "created_at": datetime.now().isoformat(),
        "events": {
            k: {
                "filename": f"Media/{v['filename']}",
                "label": WINDOWS_SYSTEM_EVENTS.get(k, {}).get("label", k),
                "reg_app": WINDOWS_SYSTEM_EVENTS.get(k, {}).get("reg_app", ".Default"),
                "prompt": v.get("prompt", ""),
            }
            for k, v in event_wav_map.items()
        },
    }

    standalone_html = (
        "<!DOCTYPE html>\n<html><head><meta charset='utf-8'>"
        f"<title>{display_name} — Windows Soundboard</title></head>"
        "<body style='background:#020617;margin:24px;'>"
        + build_audition_soundboard_html(display_name, event_wav_map, theme_name, model_label, embed_base64=False)
        + "</body></html>"
    )

    readme_txt = (
        f"====================================================================\n"
        f"  {display_name} — Windows 10 / 11 Sound Event Pack\n"
        f"  Generated with: {model_label}\n"
        f"  Theme Preset:   {theme_name}\n"
        f"====================================================================\n\n"
        f"PREVIEW IN BROWSER FIRST:\n"
        f"  Open 'audition_board.html' in any browser to audition all {len(event_wav_map)} events!\n\n"
        f"OPTION 1: 1-Click PowerShell Install (Recommended, No Admin Required)\n"
        f"--------------------------------------------------------------------\n"
        f"1. Extract this entire ZIP archive to any folder.\n"
        f"2. Right-click 'Install_{safe_id}.ps1' -> 'Run with PowerShell'.\n"
        f"3. The script copies the .wav files to %LOCALAPPDATA%\\WindowsSoundSchemes\\{safe_id},\n"
        f"   registers '{display_name}' in the Windows Sound Control Panel, activates it,\n"
        f"   and opens mmsys.cpl so you can preview or customize individual events!\n"
        f"4. To revert at any time, right-click 'Uninstall_{safe_id}.ps1' -> 'Run with PowerShell'.\n\n"
        f"OPTION 2: Manual Registry (.reg) Import\n"
        f"--------------------------------------------------------------------\n"
        f"1. Copy the 'Media' folder contents into 'C:\\Windows\\Media\\{safe_id}\\'.\n"
        f"2. Double-click 'Install_{safe_id}.reg' to register the sound scheme.\n"
    )

    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for ev_key, wav_info in event_wav_map.items():
            zf.write(wav_info["path"], arcname=f"Media/{wav_info['filename']}")
        zf.writestr(f"Install_{safe_id}.ps1", "\r\n".join(ps1_lines))
        zf.writestr(f"Uninstall_{safe_id}.ps1", "\r\n".join(uninstall_ps1))
        zf.writestr(f"Install_{safe_id}.reg", "\r\n".join(reg_lines))
        zf.writestr("Scheme.ini", "\r\n".join(ini_lines))
        zf.writestr("soundpack_manifest.json", json.dumps(manifest_data, indent=2))
        zf.writestr("audition_board.html", standalone_html)
        zf.writestr("README_INSTALL.txt", readme_txt.replace("\n", "\r\n"))

    if event_wav_map:
        first_entry = next(iter(event_wav_map.values()))
        if isinstance(first_entry, dict) and "path" in first_entry:
            LAST_WINPACK_STATE["pack_dir"] = os.path.dirname(first_entry["path"])
    LAST_WINPACK_STATE["scheme_name"] = display_name
    LAST_WINPACK_STATE["theme_name"] = theme_name
    LAST_WINPACK_STATE["model_label"] = model_label
    LAST_WINPACK_STATE["output_dir"] = output_dir
    LAST_WINPACK_STATE["event_wav_map"] = dict(event_wav_map)
    return zip_path


def finalize_rerolled_event_in_pack(
    ev_key: str,
    mastered_wav: np.ndarray,
    sr: int,
    ev_prompt: str,
    output_dir: str,
    scheme_name: str = "",
    theme_name: str = "",
    model_label: str = "",
):
    """Save a re-rolled single Windows event .wav into the active pack state, re-bundle the ZIP, and refresh the soundboard."""
    pack_dir = LAST_WINPACK_STATE.get("pack_dir")
    if not pack_dir or not os.path.exists(pack_dir):
        pack_dir = os.path.join(output_dir, f"winpack_reroll_{int(datetime.now().timestamp())}")
        os.makedirs(pack_dir, exist_ok=True)
        LAST_WINPACK_STATE["pack_dir"] = pack_dir

    fname = f"{ev_key}.wav"
    fpath = os.path.join(pack_dir, fname)
    save_pcm16_wav(fpath, mastered_wav, sr)

    ev_map = dict(LAST_WINPACK_STATE.get("event_wav_map") or {})
    ev_map[ev_key] = {"path": fpath, "filename": fname, "prompt": ev_prompt}
    LAST_WINPACK_STATE["event_wav_map"] = ev_map

    s_name = (scheme_name or LAST_WINPACK_STATE.get("scheme_name") or "AEI_Custom_Scheme").strip()
    t_name = (theme_name or LAST_WINPACK_STATE.get("theme_name") or "Custom Theme").strip()
    m_lbl = (model_label or LAST_WINPACK_STATE.get("model_label") or "AEI Sound Studio").strip()

    zip_path = package_windows_sound_scheme_zip(
        scheme_name=s_name,
        event_wav_map=ev_map,
        theme_name=t_name,
        model_label=m_lbl,
        output_dir=output_dir,
    )
    LAST_WINPACK_STATE["pack_dir"] = pack_dir
    board_html = build_audition_soundboard_html(
        scheme_name=s_name,
        event_wav_map=ev_map,
        theme_name=t_name,
        model_label=m_lbl,
        embed_base64=True,
    )
    spec_img = render_waveform_and_spectrogram(fpath, title=f"Re-Rolled Event: {ev_key}")
    status = (
        f"🔄 Re-rolled **`{ev_key}`** ({WINDOWS_SYSTEM_EVENTS.get(ev_key, {}).get('label', ev_key)}) and updated "
        f"**`{os.path.basename(zip_path)}`** ({len(ev_map)} total events in pack)!"
    )
    return zip_path, fpath, spec_img, board_html, status


SFX_CATEGORY_PROMPTS = {
    "🪟 UI / System Notification Chime": (
        "Crisp modern glass two-note ascending notification chime, warm subtle harmonic tail, "
        "clean studio interface sound effect, zero background noise"
    ),
    "🔌 Hardware Connect / Sci-Fi Telemetry": (
        "Futuristic holographic HUD confirmation tone, bright crystalline FM synth ping followed by "
        "a soft sub-bass lock click, clean stereo UI sound"
    ),
    "🗑️ Recycle Bin / Whoosh Dissolve": (
        "Satisfying airy paper and crystal particle dissolve whoosh, crisp stereo movement, "
        "clean foley UI sound effect"
    ),
    "⚔️ Cinematic Impact / Sub Drop": (
        "Massive cinematic metallic impact hit with deep sub-bass boom and dark shimmering reverb tail"
    ),
    "👣 Organic Foley / Footsteps on Wood": (
        "Crisp close-mic foley of leather boots walking slowly across a resonant hollow wooden stage"
    ),
    "🌧️ Weather / Thunderstorm Ambience": (
        "Heavy rain pattering on a glass window with a distant rolling thunderclap in wide stereo"
    ),
    "🕹️ Retro 16-Bit Coin / Power-Up": (
        "Bright nostalgic 16-bit arcade power-up arpeggio chime, crisp square and triangle synth waves"
    ),
    "🐉 Creature / Mechanical Servo": (
        "High-tech robotic exoskeleton hydraulic servo whine and heavy metallic latch locking into place"
    ),
}


class StableAudio3Studio:
    """Unified manager for Stable Audio 3 ('small-sfx', 'medium', '-base') with Diffusers fallback."""

    def __init__(self):
        self.model = None
        self.pipe_fallback = None
        self.current_model_name = None
        self.backend_type = None
        self.sample_rate = 44100
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

    def unload(self):
        self.model = None
        self.pipe_fallback = None
        self.current_model_name = None
        self.backend_type = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    def authenticate_hf(self, hf_token: str = "") -> str:
        tok = (hf_token or os.environ.get("HF_TOKEN", "")).strip()
        if tok:
            try:
                os.environ["HF_TOKEN"] = tok
                login(token=tok, add_to_git_credential=False)
                return "🔑 Authenticated with Hugging Face token."
            except Exception as e:
                return f"⚠️ HF login warning: {e}"
        return "ℹ️ Using cached HF credentials."

    def ensure_loaded(self, model_choice: str, hf_token: str = "", use_half: bool = True) -> str:
        target = model_choice.split(" ")[0].strip()
        if self.current_model_name == target and (self.model is not None or self.pipe_fallback is not None):
            return f"✅ Using cached `{target}` ({self.backend_type}) on `{self.device}`."

        self.unload()
        auth_msg = self.authenticate_hf(hf_token)

        if target in ("stable-audio-open-small", "stable-audio-open-1.0"):
            from diffusers import StableAudioPipeline
            repo_id = f"stabilityai/{target}"
            dtype = torch.float16 if (self.device == "cuda" and use_half) else torch.float32
            self.pipe_fallback = StableAudioPipeline.from_pretrained(repo_id, torch_dtype=dtype).to(self.device)
            self.backend_type = "diffusers"
            self.current_model_name = target
            self.sample_rate = int(getattr(self.pipe_fallback.vae, "sampling_rate", 44100))
            return f"✅ Loaded `{repo_id}` via Diffusers on `{self.device}` ({auth_msg})"

        try:
            from stable_audio_3 import StableAudioModel
            self.model = StableAudioModel.from_pretrained(
                target, device=self.device, model_half=bool(use_half and self.device == "cuda")
            )
            self.backend_type = "sa3"
            self.current_model_name = target
            self.sample_rate = int(getattr(self.model.model, "sample_rate", 44100))
            return f"✅ Loaded Stable Audio 3 `{target}` on `{self.device}` ({auth_msg})"
        except Exception as e:
            err_str = str(e)
            raise RuntimeError(
                f"Could not load Stable Audio 3 `{target}`: {err_str}\n\n"
                f"💡 Tip: `stabilityai/stable-audio-3-*` repositories require accepting the community license on "
                f"Hugging Face and pasting your `HF_TOKEN` in the Model Settings bar above."
            )

    def generate_audio(
        self,
        model_choice: str,
        prompt: str,
        negative_prompt: str = "",
        duration: float = 5.0,
        steps: int = 8,
        cfg_scale: float = 1.0,
        seed: int = -1,
        batch_size: int = 1,
        chunked_decode: bool = True,
        init_audio_path: str = None,
        init_noise_level: float = 0.85,
        inpaint_audio_path: str = None,
        inpaint_ranges_str: str = "",
        hf_token: str = "",
    ):
        status = self.ensure_loaded(model_choice, hf_token=hf_token)
        actual_seed = int(seed) if int(seed) >= 0 else int(np.random.randint(1, 999999))

        if self.backend_type == "diffusers":
            gen = torch.Generator(device=self.device).manual_seed(actual_seed)
            out = self.pipe_fallback(
                prompt=prompt,
                negative_prompt=negative_prompt or None,
                num_inference_steps=max(int(steps), 10),
                guidance_scale=max(float(cfg_scale), 1.0),
                audio_end_in_s=float(duration),
                num_waveforms_per_prompt=int(batch_size),
                generator=gen,
            ).audios
            tensors = [out[i].detach().cpu().float().numpy().T for i in range(out.shape[0])]
            return tensors, self.sample_rate, actual_seed, status

        kwargs = dict(
            prompt=prompt,
            negative_prompt=negative_prompt if negative_prompt and negative_prompt.strip() else None,
            duration=float(duration),
            steps=int(steps),
            cfg_scale=float(cfg_scale),
            seed=actual_seed,
            batch_size=int(batch_size),
            chunked_decode=bool(chunked_decode),
            duration_padding_sec=min(2.0, max(0.5, float(duration) * 0.5)),
        )

        if init_audio_path and os.path.exists(init_audio_path):
            wav_np, sr_in = load_audio_file_safe(init_audio_path)
            kwargs["init_audio"] = (sr_in, torch.from_numpy(wav_np))
            kwargs["init_noise_level"] = float(init_noise_level)

        if inpaint_audio_path and os.path.exists(inpaint_audio_path) and inpaint_ranges_str.strip():
            wav_np, sr_in = load_audio_file_safe(inpaint_audio_path)
            kwargs["inpaint_audio"] = (sr_in, torch.from_numpy(wav_np))
            starts, ends = [], []
            for part in inpaint_ranges_str.split(","):
                if "-" in part:
                    s_str, e_str = part.strip().split("-", 1)
                    starts.append(float(s_str))
                    ends.append(float(e_str))
            if len(starts) == 1:
                kwargs["inpaint_mask_start_seconds"] = starts[0]
                kwargs["inpaint_mask_end_seconds"] = ends[0]
            elif len(starts) > 1:
                kwargs["inpaint_mask_start_seconds"] = starts
                kwargs["inpaint_mask_end_seconds"] = ends

        audio_out = self.model.generate(**kwargs)
        if isinstance(audio_out, tuple):
            audio_out = audio_out[0]
        audio_np = audio_out.detach().cpu().float().numpy()
        if audio_np.ndim == 2:
            audio_np = audio_np[None, ...]
        results = [np.transpose(audio_np[i], (1, 0)) for i in range(audio_np.shape[0])]
        return results, self.sample_rate, actual_seed, status


STUDIO = StableAudio3Studio()
print("✅ Step 3 Complete: StableAudio3Studio & Windows Sound Packager initialized!")

In [ ]:
#@title Step 4: Launch Stable Audio 3 Gradio Studio (5 Tabs + Windows Sound Packager)
import os
import time
import gradio as gr
from IPython.display import clear_output

MODEL_CHOICES = [
    "small-sfx (433M Post-Trained 8-Step SFX — Fast, 1.7 GB VRAM)",
    "medium (1.4B Post-Trained 8-Step Stereo — Highest Quality, 5.1 GB VRAM)",
    "small-music (433M Post-Trained 8-Step Music & Stingers)",
    "small-sfx-base (433M Base 50-Step — Full CFG & Negative Prompt Control)",
    "medium-base (1.4B Base 50-Step — Full CFG & Negative Prompt Control)",
    "stable-audio-open-small (341M Diffusers Short SFX — Up to 11s)",
    "stable-audio-open-1.0 (1.0B Diffusers Stereo SFX — Up to 47s)",
]


def ui_generate_sfx(
    model_choice,
    hf_token,
    prompt,
    negative_prompt,
    duration,
    steps,
    cfg_scale,
    seed,
    batch_size,
    chunked_decode,
    apply_ui_mastering,
    trim_db,
    fade_out_ms,
):
    try:
        if not prompt or not prompt.strip():
            return None, None, None, "❌ Please enter a text prompt."
        t0 = time.time()
        wavs, sr, used_seed, load_msg = STUDIO.generate_audio(
            model_choice=model_choice,
            prompt=prompt.strip(),
            negative_prompt=negative_prompt,
            duration=float(duration),
            steps=int(steps),
            cfg_scale=float(cfg_scale),
            seed=int(seed),
            batch_size=int(batch_size),
            chunked_decode=bool(chunked_decode),
            hf_token=hf_token,
        )
        saved_paths = []
        ts = int(time.time())
        for idx, w in enumerate(wavs):
            if apply_ui_mastering:
                w = master_ui_sound(
                    w, sr, target_duration_s=float(duration), trim_silence=True,
                    trim_db=float(trim_db), fade_in_ms=5.0, fade_out_ms=float(fade_out_ms),
                )
            else:
                peak = float(np.max(np.abs(w)))
                if peak > 1e-6:
                    w = np.clip(w / peak * 0.92, -0.99, 0.99)
            out_file = os.path.join(OUTPUT_DIR, f"sa3_sfx_{ts}_seed{used_seed}_v{idx+1}.wav")
            save_pcm16_wav(out_file, w, sr)
            saved_paths.append(out_file)

        spec_img = render_waveform_and_spectrogram(saved_paths[0], title="Stable Audio 3 Output (Take 1)")
        elapsed = time.time() - t0
        second_wav = saved_paths[1] if len(saved_paths) > 1 else None
        status = (
            f"✅ Generated **{len(saved_paths)}** stereo `.wav` file(s) at **{sr} Hz** in **{elapsed:.2f}s** "
            f"(Seed: `{used_seed}`).\n- {load_msg}\n- Saved to: `{saved_paths[0]}`"
        )
        return saved_paths[0], second_wav, spec_img, status
    except Exception as e:
        return None, None, None, f"❌ Generation Error: {e}"


def ui_audio_to_audio(
    model_choice,
    hf_token,
    init_audio_path,
    init_noise_level,
    prompt,
    negative_prompt,
    duration,
    steps,
    cfg_scale,
    seed,
    apply_ui_mastering,
):
    try:
        if not init_audio_path or not os.path.exists(init_audio_path):
            return None, None, "❌ Please upload or record a source audio file first."
        if not prompt or not prompt.strip():
            return None, None, "❌ Please enter a target style prompt."
        t0 = time.time()
        wavs, sr, used_seed, load_msg = STUDIO.generate_audio(
            model_choice=model_choice,
            prompt=prompt.strip(),
            negative_prompt=negative_prompt,
            duration=float(duration),
            steps=int(steps),
            cfg_scale=float(cfg_scale),
            seed=int(seed),
            batch_size=1,
            init_audio_path=init_audio_path,
            init_noise_level=float(init_noise_level),
            hf_token=hf_token,
        )
        w = wavs[0]
        if apply_ui_mastering:
            w = master_ui_sound(w, sr, target_duration_s=float(duration))
        out_file = os.path.join(OUTPUT_DIR, f"sa3_a2a_{int(time.time())}_seed{used_seed}.wav")
        save_pcm16_wav(out_file, w, sr)
        spec_img = render_waveform_and_spectrogram(out_file, title=f"Audio-to-Audio (noise={init_noise_level})")
        return out_file, spec_img, f"✅ Audio-to-Audio variation completed in **{time.time()-t0:.2f}s** (`{out_file}`)"
    except Exception as e:
        return None, None, f"❌ Audio-to-Audio Error: {e}"


def ui_inpaint_audio(
    model_choice,
    hf_token,
    inpaint_audio_path,
    inpaint_ranges_str,
    prompt,
    duration,
    steps,
    cfg_scale,
    seed,
):
    try:
        if not inpaint_audio_path or not os.path.exists(inpaint_audio_path):
            return None, None, "❌ Please upload an audio file to inpaint or extend."
        if not inpaint_ranges_str or "-" not in inpaint_ranges_str:
            return None, None, "❌ Please specify at least one time range in seconds (e.g. `1.5-3.5` or `1.0-2.0, 4.0-5.5`)."
        t0 = time.time()
        wavs, sr, used_seed, _ = STUDIO.generate_audio(
            model_choice=model_choice,
            prompt=prompt.strip() or "clean sound effect continuation",
            duration=float(duration),
            steps=int(steps),
            cfg_scale=float(cfg_scale),
            seed=int(seed),
            batch_size=1,
            inpaint_audio_path=inpaint_audio_path,
            inpaint_ranges_str=inpaint_ranges_str,
            hf_token=hf_token,
        )
        out_file = os.path.join(OUTPUT_DIR, f"sa3_inpaint_{int(time.time())}_seed{used_seed}.wav")
        save_pcm16_wav(out_file, wavs[0], sr)
        spec_img = render_waveform_and_spectrogram(out_file, title=f"Inpainted Regions: {inpaint_ranges_str}s")
        return out_file, spec_img, f"✅ Inpainting/Continuation finished in **{time.time()-t0:.2f}s** (`{out_file}`)"
    except Exception as e:
        return None, None, f"❌ Inpainting Error: {e}"


def ui_build_windows_sound_pack(
    model_choice,
    hf_token,
    scheme_name,
    theme_name,
    custom_style,
    selected_events,
    steps,
    cfg_scale,
    base_seed,
    trim_db,
    fade_out_ms,
):
    try:
        if not selected_events:
            return None, None, None, "", "❌ Please select at least one Windows system event."
        t0 = time.time()
        pack_dir = os.path.join(OUTPUT_DIR, f"winpack_{int(time.time())}")
        os.makedirs(pack_dir, exist_ok=True)
        event_wav_map = {}
        preview_wavs = []
        seed_base = int(base_seed) if int(base_seed) >= 0 else int(np.random.randint(1000, 900000))

        for idx, ev_key in enumerate(selected_events):
            ev_meta = WINDOWS_SYSTEM_EVENTS.get(ev_key, WINDOWS_SYSTEM_EVENTS["SystemNotification"])
            ev_prompt = build_windows_event_prompt(ev_key, theme_name, custom_style)
            dur = float(ev_meta["duration"])
            wavs, sr, _, _ = STUDIO.generate_audio(
                model_choice=model_choice,
                prompt=ev_prompt,
                duration=max(dur + 0.5, 1.5),
                steps=int(steps),
                cfg_scale=float(cfg_scale),
                seed=seed_base + idx,
                batch_size=1,
                hf_token=hf_token,
            )
            mastered = master_ui_sound(
                wavs[0], sr, target_duration_s=dur, trim_silence=True,
                trim_db=float(trim_db), fade_in_ms=5.0, fade_out_ms=float(fade_out_ms),
            )
            fname = f"{ev_key}.wav"
            fpath = os.path.join(pack_dir, fname)
            save_pcm16_wav(fpath, mastered, sr)
            event_wav_map[ev_key] = {"path": fpath, "filename": fname, "prompt": ev_prompt}
            preview_wavs.append(fpath)

        model_lbl = f"Stable Audio 3 ({model_choice.split(' ')[0]})"
        zip_path = package_windows_sound_scheme_zip(
            scheme_name=scheme_name,
            event_wav_map=event_wav_map,
            theme_name=theme_name,
            model_label=model_lbl,
            output_dir=OUTPUT_DIR,
        )
        board_html = build_audition_soundboard_html(
            scheme_name=scheme_name,
            event_wav_map=event_wav_map,
            theme_name=theme_name,
            model_label=model_lbl,
            embed_base64=True,
        )
        first_wav = preview_wavs[0] if preview_wavs else None
        spec_img = render_waveform_and_spectrogram(first_wav, title=f"Windows Event Preview: {selected_events[0]}") if first_wav else None
        summary = (
            f"✅ Built Windows Sound Event Pack **`{os.path.basename(zip_path)}`** with **{len(event_wav_map)}** events "
            f"in **{time.time()-t0:.1f}s**!\n"
            f"- Includes `Media/*.wav` (44.1 kHz 16-bit PCM), `Install_*.ps1`, `Uninstall_*.ps1`, `Install_*.reg`, and `audition_board.html`."
        )
        return zip_path, first_wav, spec_img, board_html, summary
    except Exception as e:
        return None, None, None, "", f"❌ Windows Sound Pack Error: {e}"


def ui_sa3_reroll_single_event(
    model_choice,
    hf_token,
    scheme_name,
    theme_name,
    custom_style,
    ev_key,
    ev_custom_prompt,
    reroll_seed,
    pitch_semitones,
    steps,
    cfg_scale,
    trim_db,
    fade_out_ms,
):
    try:
        ev_meta = WINDOWS_SYSTEM_EVENTS.get(ev_key, WINDOWS_SYSTEM_EVENTS["SystemNotification"])
        dur = float(ev_meta["duration"])
        ev_prompt = (ev_custom_prompt or "").strip() or build_windows_event_prompt(ev_key, theme_name, custom_style)
        wavs, sr, _, _ = STUDIO.generate_audio(
            model_choice=model_choice,
            prompt=ev_prompt,
            duration=max(dur + 0.5, 1.5),
            steps=int(steps),
            cfg_scale=float(cfg_scale),
            seed=int(reroll_seed),
            batch_size=1,
            hf_token=hf_token,
        )
        mastered = master_ui_sound(
            wavs[0], sr, target_duration_s=dur, trim_silence=True,
            trim_db=float(trim_db), fade_in_ms=1.5, fade_out_ms=float(fade_out_ms),
            pitch_semitones=float(pitch_semitones),
        )
        return finalize_rerolled_event_in_pack(
            ev_key=ev_key,
            mastered_wav=mastered,
            sr=sr,
            ev_prompt=ev_prompt,
            output_dir=OUTPUT_DIR,
            scheme_name=scheme_name,
            theme_name=theme_name,
            model_label=f"Stable Audio 3 ({model_choice.split(' ')[0]})",
        )
    except Exception as e:
        return None, None, None, "", f"❌ Single-Event Re-Roll Error: {e}"


def ui_apply_lora(lora_path, lora_strength):
    try:
        if STUDIO.model is None or STUDIO.backend_type != "sa3":
            return "⚠️ Load a native Stable Audio 3 model (`small-sfx` or `medium`) first before attaching a LoRA."
        if not lora_path or not os.path.exists(lora_path.strip()):
            return "❌ LoRA file path not found."
        STUDIO.model.load_lora([lora_path.strip()])
        STUDIO.model.set_lora_strength(float(lora_strength))
        return f"✅ Loaded LoRA `{os.path.basename(lora_path.strip())}` at strength `{lora_strength:.2f}`!"
    except Exception as e:
        return f"❌ LoRA Error: {e}"


def welcome_status():
    gpu_str = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU Mode"
    return f"🟢 **Stable Audio 3 Studio Ready** on `{gpu_str}` — Select `small-sfx` (fast, 1.7 GB VRAM) or `medium` (highest stereo fidelity) above."


with gr.Blocks(title="Stable Audio 3 — SFX & Windows Sound Pack Studio", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "## 🔊 Stable Audio 3 — Stereo Sound Effects, Editing & Windows Sound Scheme Studio\n"
        "Generate **44.1 kHz stereo** sound effects in 8 steps with `small-sfx` / `medium`, morph existing clips, "
        "inpaint multi-region audio, or export ready-to-install **Windows 10/11 Sound Event Packs**."
    )
    status_banner = gr.Markdown("⏳ Initializing studio...")

    with gr.Row():
        model_dropdown = gr.Dropdown(
            choices=MODEL_CHOICES,
            value=MODEL_CHOICES[0],
            label="🎛️ Stable Audio 3 Checkpoint",
            info="Use 'small-sfx' (1.7 GB VRAM, 8 steps) for fast UI/SFX or 'medium' (5.1 GB VRAM) for full SAME-L stereo depth.",
            scale=3,
        )
        hf_token_box = gr.Textbox(
            value=os.environ.get("HF_TOKEN", ""),
            type="password",
            label="🔑 Hugging Face Token (Required for Gated stabilityai/* Repos)",
            placeholder="hf_... (or set HF_TOKEN in Colab Secrets)",
            info="Accept the Stability AI Community License on Hugging Face, then paste your read token here.",
            scale=2,
        )

    with gr.Tabs():
        with gr.Tab("🔊 1. Text-to-SFX Studio"):
            with gr.Row():
                with gr.Column(scale=3):
                    preset_cat = gr.Dropdown(
                        choices=list(SFX_CATEGORY_PROMPTS.keys()),
                        value=list(SFX_CATEGORY_PROMPTS.keys())[0],
                        label="✨ Quick SFX Category Preset",
                        info="Select a category to populate a proven acoustic prompt below.",
                    )
                    t1_prompt = gr.Textbox(
                        value=list(SFX_CATEGORY_PROMPTS.values())[0],
                        lines=3,
                        label="📝 Sound Effect Prompt",
                        info="Describe the sound source, material, pitch contour, and acoustic space.",
                    )
                    t1_neg = gr.Textbox(
                        value="low quality, harsh distortion, background hiss, hum, clipping",
                        label="🚫 Negative Prompt (Active on '-base' & Diffusers Models)",
                        info="Steers base models away from unwanted artifacts.",
                    )
                    with gr.Row():
                        t1_dur = gr.Slider(0.5, 60.0, value=3.0, step=0.5, label="⏱️ Duration (Seconds)", info="Short (0.5–3s) for UI sounds; up to 60s for ambiences.")
                        t1_steps = gr.Slider(4, 60, value=8, step=1, label="⚡ Sampling Steps", info="Use 8 steps for post-trained models ('small-sfx', 'medium') or 50 for '-base'.")
                        t1_cfg = gr.Slider(1.0, 10.0, value=1.0, step=0.5, label="🧭 CFG Scale", info="Use 1.0 for post-trained models; use 5.0–7.0 for '-base' models.")
                    with gr.Row():
                        t1_seed = gr.Number(value=-1, precision=0, label="🎲 Seed (-1 = Random)", info="Set a fixed integer for reproducible sound design.")
                        t1_batch = gr.Slider(1, 2, value=2, step=1, label="📦 Batch Variations", info="Generate 1 or 2 takes simultaneously to compare.")
                        t1_chunked = gr.Checkbox(value=True, label="🧩 Chunked VAE Decode", info="Reduces peak VRAM during SAME autoencoder decoding.")
                    with gr.Accordion("🛠️ Zero-Latency UI Mastering & Trim Controls", open=False):
                        with gr.Row():
                            t1_master = gr.Checkbox(value=True, label="✨ Auto-Trim Leading Silence & Apply Click-Free Fades", info="Removes pre-attack dead air so UI sounds trigger instantaneously.")
                            t1_trim_db = gr.Slider(-60.0, -25.0, value=-42.0, step=1.0, label="✂️ Silence Threshold (dB)", info="Samples quieter than this before the first transient are trimmed.")
                            t1_fade_out = gr.Slider(20.0, 400.0, value=120.0, step=10.0, label="📉 Tail Fade-Out (ms)", info="Smoothly tapers the end of the clip so it never clicks.")
                    t1_btn = gr.Button("🚀 Generate Stereo Sound Effect", variant="primary")

                with gr.Column(scale=2):
                    t1_audio1 = gr.Audio(label="🎧 Generated Take 1 (44.1 kHz Stereo PCM)", type="filepath")
                    t1_audio2 = gr.Audio(label="🎧 Generated Take 2 (Variation)", type="filepath")
                    t1_spec = gr.Image(label="📊 Waveform & Spectrogram Inspector", type="filepath")
                    t1_status = gr.Markdown()

            preset_cat.change(fn=lambda k: SFX_CATEGORY_PROMPTS.get(k, ""), inputs=[preset_cat], outputs=[t1_prompt])
            t1_btn.click(
                fn=ui_generate_sfx,
                inputs=[model_dropdown, hf_token_box, t1_prompt, t1_neg, t1_dur, t1_steps, t1_cfg, t1_seed, t1_batch, t1_chunked, t1_master, t1_trim_db, t1_fade_out],
                outputs=[t1_audio1, t1_audio2, t1_spec, t1_status],
            )

        with gr.Tab("🔄 2. Audio-to-Audio Style Variation"):
            with gr.Row():
                with gr.Column():
                    t2_init_audio = gr.Audio(label="📂 Upload Reference Audio (.wav / .mp3)", type="filepath")
                    t2_noise = gr.Slider(0.05, 1.0, value=0.75, step=0.05, label="🌊 Init Noise Level (0.1 = Subtle Variation, 0.85 = Strong Restyle)", info="Controls how much of the original audio structure is retained vs re-imagined.")
                    t2_prompt = gr.Textbox(value="crystalline futuristic glass marimba UI notification chime, clean stereo", label="📝 Target Style Prompt", info="Describe the new material, instrument, or mood to apply.")
                    t2_neg = gr.Textbox(value="distortion, noise, mud", label="🚫 Negative Prompt", info="Qualities to avoid.")
                    with gr.Row():
                        t2_dur = gr.Slider(0.5, 30.0, value=3.0, step=0.5, label="⏱️ Output Duration (s)", info="Target duration in seconds.")
                        t2_steps = gr.Slider(4, 50, value=8, step=1, label="⚡ Steps", info="8 for post-trained; 50 for base.")
                        t2_cfg = gr.Slider(1.0, 9.0, value=1.0, step=0.5, label="🧭 CFG Scale", info="Guidance scale.")
                        t2_seed = gr.Number(value=-1, precision=0, label="🎲 Seed", info="-1 for random.")
                    t2_master = gr.Checkbox(value=True, label="✨ Apply Zero-Latency UI Mastering", info="Auto-trim leading silence and normalize peak.")
                    t2_btn = gr.Button("🔄 Restyle Audio", variant="primary")
                with gr.Column():
                    t2_out_audio = gr.Audio(label="🎧 Restyled Audio Output", type="filepath")
                    t2_spec = gr.Image(label="📊 Waveform & Spectrogram", type="filepath")
                    t2_status = gr.Markdown()

            t2_btn.click(
                fn=ui_audio_to_audio,
                inputs=[model_dropdown, hf_token_box, t2_init_audio, t2_noise, t2_prompt, t2_neg, t2_dur, t2_steps, t2_cfg, t2_seed, t2_master],
                outputs=[t2_out_audio, t2_spec, t2_status],
            )

        with gr.Tab("🩹 3. Multi-Region Inpainting & Continuation"):
            with gr.Row():
                with gr.Column():
                    t3_audio = gr.Audio(label="📂 Source Audio File", type="filepath")
                    t3_ranges = gr.Textbox(
                        value="1.0-2.5",
                        label="⏱️ Mask Time Regions in Seconds (Comma-Separated for Multi-Region)",
                        info="Example: '1.0-2.5' for a single window, or '0.5-1.2, 2.5-3.5' for multiple non-contiguous regions. Set start >= clip length to extend/continue!",
                    )
                    t3_prompt = gr.Textbox(value="bright metallic shimmer flourish transition", label="📝 Inpaint / Continuation Prompt", info="Describe what should fill the masked region(s).")
                    with gr.Row():
                        t3_dur = gr.Slider(1.0, 60.0, value=4.0, step=0.5, label="⏱️ Total Output Duration (s)", info="Choose a duration longer than source audio to perform continuation.")
                        t3_steps = gr.Slider(4, 50, value=8, step=1, label="⚡ Steps", info="Sampling steps.")
                        t3_cfg = gr.Slider(1.0, 9.0, value=1.0, step=0.5, label="🧭 CFG Scale", info="Guidance scale.")
                        t3_seed = gr.Number(value=-1, precision=0, label="🎲 Seed", info="Random seed.")
                    t3_btn = gr.Button("🩹 Inpaint / Extend Audio", variant="primary")
                with gr.Column():
                    t3_out = gr.Audio(label="🎧 Inpainted / Extended Audio", type="filepath")
                    t3_spec = gr.Image(label="📊 Spectrogram Preview", type="filepath")
                    t3_status = gr.Markdown()

            t3_btn.click(
                fn=ui_inpaint_audio,
                inputs=[model_dropdown, hf_token_box, t3_audio, t3_ranges, t3_prompt, t3_dur, t3_steps, t3_cfg, t3_seed],
                outputs=[t3_out, t3_spec, t3_status],
            )

        with gr.Tab("🪟 4. Windows Sound Event Pack Builder"):
            gr.Markdown(
                "### 🪟 Windows 10/11 Sound Scheme Generator (`HKEY_CURRENT_USER\\AppEvents\\Schemes`)\n"
                "Batch-generate a cohesive stereo Windows sound pack with **zero-latency attack trimming**, "
                "**16-bit PCM `.wav` mastering**, and an automated **PowerShell (`Install_*.ps1`) + Registry (`Install_*.reg`) installer**."
            )
            with gr.Row():
                with gr.Column(scale=3):
                    with gr.Row():
                        w_scheme = gr.Textbox(value="AeroGlass_SA3", label="🏷️ Sound Scheme Name", info="Display name registered in Windows Sound Control Panel (mmsys.cpl).")
                        w_theme = gr.Dropdown(
                            choices=list(SONIC_THEME_PRESETS.keys()),
                            value=list(SONIC_THEME_PRESETS.keys())[0],
                            label="🎨 Sonic Theme Preset",
                            info="Cohesive acoustic aesthetic applied across every system event in the pack.",
                        )
                    w_custom_style = gr.Textbox(
                        value="",
                        placeholder="Optional: override theme style description (e.g., 'warm cello harmonics and soft wooden clicks')...",
                        label="✍️ Custom Theme Style Override (Optional)",
                        info="Leave blank to use the selected Sonic Theme Preset above.",
                    )
                    w_events = gr.CheckboxGroup(
                        choices=list(WINDOWS_SYSTEM_EVENTS.keys()),
                        value=DEFAULT_SELECTED_WIN_EVENTS,
                        label="🔔 Windows System Events to Generate (16 Standard AppEvents)",
                        info="Select which Windows events to include in your downloadable sound pack.",
                    )
                    with gr.Row():
                        w_steps = gr.Slider(4, 50, value=8, step=1, label="⚡ Steps per Event", info="8 steps for 'small-sfx' / 'medium'.")
                        w_cfg = gr.Slider(1.0, 9.0, value=1.0, step=0.5, label="🧭 CFG Scale", info="1.0 for post-trained models.")
                        w_seed = gr.Number(value=42, precision=0, label="🎲 Base Seed", info="Deterministic seed offset per event.")
                    with gr.Row():
                        w_trim_db = gr.Slider(-60.0, -25.0, value=-42.0, step=1.0, label="✂️ Zero-Latency Trim Threshold (dB)", info="Strips pre-attack silence so system events trigger with 0ms lag.")
                        w_fade_out = gr.Slider(40.0, 300.0, value=120.0, step=10.0, label="📉 Tail Fade-Out (ms)", info="Ensures click-free release on every Windows event.")
                    w_btn = gr.Button("🪟 Build Complete Windows Sound Scheme (.zip + .ps1 + .reg)", variant="primary")
                    with gr.Accordion("🎯 Single-Event Re-Roll & Pitch Customizer (Update 1 Event Without Regenerating Pack)", open=False):
                        with gr.Row():
                            w_rr_ev = gr.Dropdown(choices=list(WINDOWS_SYSTEM_EVENTS.keys()), value="DeviceConnect", label="🔔 Event to Re-Roll / Replace", info="Pick a single Windows event to regenerate and swap into the current pack.")
                            w_rr_seed = gr.Number(value=-1, precision=0, label="🎲 New Seed (-1 = Random)", info="Fresh seed for this event.")
                            w_rr_pitch = gr.Slider(-6.0, 6.0, value=0.0, step=0.5, label="🎵 Pitch Transpose (Semitones)", info="Transpose this event up/down in semitones.")
                        w_rr_prompt = gr.Textbox(value="", placeholder="Optional custom prompt for this single event (leave blank to use theme prompt)...", label="✍️ Single-Event Prompt Override (Optional)", info="Override prompt just for this event.")
                        w_rr_btn = gr.Button("🔄 Re-Roll Single Event & Update Pack (.zip + Soundboard)")

                with gr.Column(scale=2):
                    w_zip = gr.File(label="📦 Download Ready-to-Install Windows Sound Pack (.zip)")
                    w_preview_audio = gr.Audio(label="🎧 Sample Generated Event Preview", type="filepath")
                    w_spec = gr.Image(label="📊 Event Transient & Spectrogram Check", type="filepath")
                    w_status = gr.Markdown()
            w_board = gr.HTML(label="🎹 Interactive Windows Sound Scheme Audition Board")

            w_btn.click(
                fn=ui_build_windows_sound_pack,
                inputs=[model_dropdown, hf_token_box, w_scheme, w_theme, w_custom_style, w_events, w_steps, w_cfg, w_seed, w_trim_db, w_fade_out],
                outputs=[w_zip, w_preview_audio, w_spec, w_board, w_status],
            )
            w_rr_btn.click(
                fn=ui_sa3_reroll_single_event,
                inputs=[model_dropdown, hf_token_box, w_scheme, w_theme, w_custom_style, w_rr_ev, w_rr_prompt, w_rr_seed, w_rr_pitch, w_steps, w_cfg, w_trim_db, w_fade_out],
                outputs=[w_zip, w_preview_audio, w_spec, w_board, w_status],
            )

        with gr.Tab("🧬 5. LoRA Adapter & VRAM Manager"):
            with gr.Row():
                with gr.Column():
                    lora_path_box = gr.Textbox(
                        value="",
                        placeholder="/content/drive/MyDrive/my_sa3_lora.ckpt",
                        label="🧬 Custom Stable Audio 3 LoRA Checkpoint Path",
                        info="Path to a fine-tuned Stable Audio 3 LoRA checkpoint.",
                    )
                    lora_str_slider = gr.Slider(0.0, 2.0, value=1.0, step=0.05, label="🎚️ LoRA Strength", info="Runtime multiplier for the loaded LoRA weights.")
                    with gr.Row():
                        lora_btn = gr.Button("🧬 Load & Apply LoRA")
                        unload_btn = gr.Button("🧹 Unload Models & Free GPU VRAM")
                with gr.Column():
                    lora_status = gr.Markdown("ℹ️ Use this tab to attach custom Stable Audio 3 LoRA adapters or free GPU memory.")

            lora_btn.click(fn=ui_apply_lora, inputs=[lora_path_box, lora_str_slider], outputs=[lora_status])
            unload_btn.click(fn=lambda: (STUDIO.unload() or "🧹 GPU VRAM cleared."), inputs=None, outputs=[lora_status])

    demo.load(fn=welcome_status, inputs=None, outputs=[status_banner])

clear_output()
demo.queue(default_concurrency_limit=1)
demo.launch(share=True, inline=False, show_error=True)

In [ ]:
#@title Step 5: Keep-Alive Helper & GPU VRAM Monitor
import time
import torch

KEEP_ALIVE_MINUTES = 0 #@param {type:"integer"}

try:
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated(0) / (1024 ** 3)
        resv = torch.cuda.memory_reserved(0) / (1024 ** 3)
        print(f"📊 GPU VRAM Status — Allocated: {alloc:.2f} GB | Reserved: {resv:.2f} GB")
    if KEEP_ALIVE_MINUTES > 0:
        print(f"⏳ Keeping Gradio session alive for {KEEP_ALIVE_MINUTES} minutes...")
        for m in range(KEEP_ALIVE_MINUTES):
            time.sleep(60)
            print(f"  ✓ Active: {m + 1}/{KEEP_ALIVE_MINUTES} min elapsed")
    else:
        print("ℹ️ Keep-alive loop skipped (KEEP_ALIVE_MINUTES=0). Set > 0 if running unattended.")
except KeyboardInterrupt:
    print("⏹️ Keep-alive stopped by user.")
except Exception as e:
    print(f"Keep-alive warning: {e}")

In [ ]:
#@title Step 6: Quick Test — Single Stereo Sound Effect Synthesis
import os
from IPython.display import Audio, FileLink, display

QUICKTEST_MODEL = "small-sfx (433M Post-Trained 8-Step SFX — Fast, 1.7 GB VRAM)" #@param ["small-sfx (433M Post-Trained 8-Step SFX — Fast, 1.7 GB VRAM)", "medium (1.4B Post-Trained 8-Step Stereo — Highest Quality, 5.1 GB VRAM)", "stable-audio-open-small (341M Diffusers Short SFX — Up to 11s)"]
QUICKTEST_PROMPT = "Crisp modern glass two-note ascending notification chime, studio UI sound effect" #@param {type:"string"}
QUICKTEST_DURATION = 2.0 #@param {type:"number"}
QUICKTEST_SEED = 42 #@param {type:"integer"}

try:
    wav_path, _, spec_path, msg = ui_generate_sfx(
        model_choice=QUICKTEST_MODEL,
        hf_token=os.environ.get("HF_TOKEN", ""),
        prompt=QUICKTEST_PROMPT,
        negative_prompt="noise, distortion",
        duration=QUICKTEST_DURATION,
        steps=8,
        cfg_scale=1.0,
        seed=QUICKTEST_SEED,
        batch_size=1,
        chunked_decode=True,
        apply_ui_mastering=True,
        trim_db=-42.0,
        fade_out_ms=120.0,
    )
    print(msg)
    if wav_path and os.path.exists(wav_path):
        display(Audio(filename=wav_path))
        display(FileLink(wav_path))
except Exception as e:
    print(f"Quick test error: {e}")

In [ ]:
#@title Step 7: Headless Batch Windows Sound Event Pack Generator
import os
from tqdm.auto import tqdm
from IPython.display import FileLink, display

BATCH_SCHEME_NAME = "AeroGlass_SA3" #@param {type:"string"}
BATCH_THEME = "Frutiger Aero Crystal Glass" #@param ["Frutiger Aero Crystal Glass", "Cyberpunk Neon HUD", "Cozy Wooden Kalimba & Mallet", "Sci-Fi Spaceship Bridge", "Retro 16-Bit Chiptune Arcade", "Warm Lo-Fi Analog Tape Synth", "Minimalist Tactile Micro-Foley", "Dark Fantasy Crystal & Harp"]
BATCH_SEED = 100 #@param {type:"integer"}

try:
    event_map = {}
    pack_dir = os.path.join(OUTPUT_DIR, f"batch_winpack_{BATCH_SEED}")
    os.makedirs(pack_dir, exist_ok=True)
    for idx, ev_key in enumerate(tqdm(DEFAULT_SELECTED_WIN_EVENTS, desc="Generating Windows Sound Pack")):
        try:
            ev_meta = WINDOWS_SYSTEM_EVENTS[ev_key]
            prompt = build_windows_event_prompt(ev_key, BATCH_THEME, "")
            wavs, sr, _, _ = STUDIO.generate_audio(
                model_choice="small-sfx",
                prompt=prompt,
                duration=max(float(ev_meta["duration"]) + 0.5, 1.5),
                steps=8,
                cfg_scale=1.0,
                seed=BATCH_SEED + idx,
                batch_size=1,
            )
            mastered = master_ui_sound(wavs[0], sr, target_duration_s=float(ev_meta["duration"]))
            fpath = os.path.join(pack_dir, f"{ev_key}.wav")
            save_pcm16_wav(fpath, mastered, sr)
            event_map[ev_key] = {"path": fpath, "filename": f"{ev_key}.wav", "prompt": prompt}
        except Exception as item_e:
            print(f"⚠️ Event {ev_key} skipped: {item_e}")

    if event_map:
        zip_out = package_windows_sound_scheme_zip(BATCH_SCHEME_NAME, event_map, BATCH_THEME, "Stable Audio 3", OUTPUT_DIR)
        print(f"✅ Packaged Windows Sound Scheme: {zip_out}")
        display(FileLink(zip_out))
except Exception as e:
    print(f"Batch Windows Pack error: {e}")